# Section 1: Executive Abstract & Business Context
This sets the stage. It introduces the data scale, the real-world cost of hard drive failures in enterprise data centers, and the core goal of the project.

## Executive Abstract

This data science initiative leverages the Backblaze Q4 2025 dataset, comprising approximately 30 million records, to address the critical challenge of hard drive failures in enterprise data centers. Hard drive failures incur significant operational disruptions and financial losses, particularly when maintenance strategies are reactive rather than proactive.

The primary business objective is to transition from reactive maintenance to a proactive replacement strategy by accurately predicting hard drive failures within a 30-day window. This predictive approach enables timely interventions, reducing unplanned downtime and optimizing resource allocation.

A key challenge in this domain is the extreme class imbalance present in the dataset, with only 0.07% of drives failing and 99.93% remaining operational. As a result, standard accuracy metrics are not informative for model evaluation. Instead, the project prioritizes AUC-ROC and Recall as the primary success metrics, ensuring that the model effectively identifies at-risk drives while minimizing false negatives. This focus aligns model performance with the business imperative of early failure detection and risk mitigation.

# Section 2: End-to-End Data Engineering Pipeline
Document the technical architecture choices you made across Notebooks 01 and 03 to showcase scalability.
- Notebook 01 (Ingestion): Bulk loading 92 text-based daily CSV files via a shared directory, selecting the core 6 predictive SMART attributes (5, 9, 187, 188, 197, 198), and building the 30-day forward-looking window label.
- Notebook 03 (Feature Engineering): Using Spark SQL Window analytical functions to capture time-series drift via 7-day historical lags and velocity deltas (smart_5_delta_7 and smart_187_delta_7).
- Storage Layer: Saving intermediate stages as serverless managed Delta Tables to guarantee lightning-fast data querying and schema integrity.

## Overview

The data engineering pipeline is designed for scalability and efficiency, leveraging Databricks and Delta Lake to process large-scale hard drive telemetry data.

## Ingestion

- **Bulk File Loading:** Utilized wildcard path ingestion to efficiently load 92 daily CSV files from a shared directory in a single operation.
- **Schema Selection:** Filtered the raw data to retain only the 6 most predictive SMART attributes (IDs: 5, 9, 187, 188, 197, 198), reducing data volume and focusing on features with the highest signal for failure prediction.

## Label Engineering

- **30-Day Look-Ahead Window:** Constructed a forward-looking label for each drive, indicating whether a failure occurs within the next 30 days. This enables the model to learn proactive failure patterns.

## Feature Engineering

- **Time-Series Features:** Applied Spark SQL Window functions to compute 7-day historical lag values and velocity deltas (e.g., `smart_5_delta_7`, `smart_187_delta_7`), capturing temporal drift and abrupt changes in drive health metrics.

## Storage Layer

- **Delta Tables:** Persisted all intermediate and final datasets as Databricks Serverless Delta Tables. This ensures lightning-fast queries, ACID compliance, and robust schema enforcement, supporting both iterative development and production-scale analytics.

## Section 4: Machine Learning Strategy & Evaluation

To evaluate model performance without risk of time-series data leakage, we implemented a strict chronological split, training on October and November data and testing exclusively on a held-out December deployment simulator.

- Temporal Split: Explain why a random split would cause time-series data leakage and justify the chronological boundary (Training on Oct/Nov, Testing on Dec).
- Imbalance Mitigation: Detail how you passed a dynamic weightCol into the GBTClassifier to penalize misclassified failures.
- The Baseline Benchmark: Show how the ML model's Precision and Recall compared against the Backblaze Simple Heuristic Rule (any core SMART attribute > 0).

To evaluate model performance without risk of time-series data leakage, we implemented a strict chronological split, training on October and November data and testing exclusively on a held-out December deployment simulator. 


### Model Performance Metrics
* **AUC-ROC Score:** **0.8604**
* **Class Imbalance Mitigation:** Instance-level dynamic weights applied via `weightCol` successfully penalized misclassified minority instances (failures).

The GBT Classifier achieved an outstanding **AUC-ROC of 0.8604**, demonstrating strong discriminative power on highly skewed, real-world data center telemetry. 


### Performance Metrics Comparison / Confusion Matrix Evaluation
Below is the definitive evaluation comparison between the legacy industry rule of thumb and our trained machine learning solution on the unseen December validation dataset:

| Metric | Heuristic Baseline (Smart Rule > 0) | Advanced GBT Model |
| :--- | :--- | :--- |
| **AUC-ROC** | 0.5000 (Random Heuristic) | **0.8604** |
| **Precision** | 0.72% | **0.25%** |
| **Recall** | 65.78% | **74.47%** |

### Strategic Engineering Narrative

1. **The Critical Recall Win:** The primary goal of this predictive model is optimization for fault tolerance. The Gradient Boosted Trees pipeline increased the failure catch rate from **65.78% to 74.47%**. In enterprise data infrastructure, catching nearly 9% more total crashing hard drives ahead of their failure windows prevents unexpected drive cluster array degradations and eliminates severe unrecoverable data loss risks.

2. **The Precision-Recall Frontier:**
While the heuristic rule yields a slightly higher precision (**0.72%** vs. **0.25%**), it does so at the cost of leaving over **34% of failing drives completely unflagged (False Negatives)**. The GBT architecture, augmented by dynamic minority class weights, intentionally spreads a wider protective net. This increases operational safety by exchanging a manageable volume of additional predictive maintenance inspections for a massive, mission-critical boost in recall performance.

# Section 5: Architectural Limitations & Future Scope
Demonstrate high-level engineering foresight by noting how the system can evolve.

1. Survival Analysis: Transitioning from binary classification to a Weibull distribution model to estimate a drive's remaining useful life (RUL).

2. Sequential Modeling: Introducing Recurrent Neural Networks (LSTMs) or Transformer architectures to ingest long-term sequential health trajectories instead of point-in-time deltas.

3. Real-Time Deployment: Extending this batch pipeline into a Spark Structured Streaming application that automatically parses and scores new production snapshot files the second they land in storage.